<a href="https://colab.research.google.com/github/mru-art/internship/blob/main/ModelComparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install rank_bm25

In [13]:
#imports
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_curve, auc, precision_recall_curve)
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt
from rank_bm25 import BM25Okapi
import os

In [14]:
output_dir = "Pictures for paper"
os.makedirs(output_dir, exist_ok=True)

THRESHOLD = 0.40      # global threshold; can tune per-model later
TOP_N     = 5         # number of top models shown in ROC/PR/Radar charts

#config and output dir

In [15]:
models = [
    # General purpose
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-MiniLM-L12-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "sentence-transformers/all-roberta-large-v1",

    # Paraphrase
    "sentence-transformers/paraphrase-MiniLM-L6-v2",
    "sentence-transformers/paraphrase-mpnet-base-v2",
    "sentence-transformers/paraphrase-albert-small-v2",
    "sentence-transformers/paraphrase-distilroberta-base-v2",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",

    # QA / semantic search optimised
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    "sentence-transformers/multi-qa-mpnet-base-dot-v1",
    "sentence-transformers/msmarco-distilbert-base-v4",
    "sentence-transformers/stsb-roberta-large",

    # Google GTR
    "sentence-transformers/gtr-t5-base",
    "sentence-transformers/gtr-t5-large",

    # BAAI BGE (strong retrieval)
    "BAAI/bge-small-en-v1.5",
    "BAAI/bge-base-en-v1.5",
    "BAAI/bge-large-en-v1.5",      # kept once (duplicate removed)

    # Microsoft E5
    "intfloat/e5-base-v2",
    "intfloat/e5-large-v2",

    # Nomic
    "nomic-ai/nomic-embed-text-v1",

    # Domain-specific (these ARE released as sentence-transformer compatible)
    "AI-Growth-Lab/PatentSBERTa",
    "allenai/scibert_scivocab_uncased",  # NOTE: needs pooling — handled below
]

#Model List  (only valid SentenceTransformer models)

In [16]:
patents = [
    "Beamforming method for millimeter wave communication in 5G networks",
    "Low power wireless communication for IoT sensor devices",
    "High throughput wireless data transmission using WiFi technology",
    "Vehicle communication using network slicing in 5G systems",
    "Beamforming method for millimeter wave communication in 5G networks",
    "Low power wireless communication for IoT sensor devices",
]

standards = [
    # --- paste the full standards list from the original script here ---
    # (abbreviated for readability in this cell template)
    "3GPP TS 38.201: NR physical layer general description",
    "3GPP TS 38.211: NR physical channels and modulation",
    "3GPP TS 38.213: NR physical layer procedures for control",
    "3GPP TS 38.214: NR physical layer procedures for data",
    "3GPP TS 38.223: NR physical layer for V2X",
    "3GPP TS 38.321: NR MAC protocol",
    "3GPP TS 38.331: NR RRC protocol",
    "3GPP TS 38.401: NG-RAN architecture",
    "3GPP TS 38.405: NR RAN slicing support",
    "3GPP TS 23.501: 5G system architecture",
    "3GPP TS 23.508: Network slicing support",
    "3GPP TS 36.404: NB-IoT coverage enhancement",
    "3GPP TS 36.405: NB-IoT power saving modes",
    "3GPP TS 36.410: LTE-M architecture",
]

# Patents and standards

In [17]:
def assign_label(patent: str, standard: str) -> int:
    p, s = patent.lower(), standard.lower()
    rules = [
        ("beamforming" in p or "millimeter wave" in p or "5g" in p,
         any(k in s for k in ["beam", "mmwave", "5g", "nr", "38.2", "38.3"])),

        ("iot" in p or "low power" in p or "sensor" in p,
         any(k in s for k in ["iot", "nb-iot", "lte-m", "power saving",
                               "36.4", "machine type", "mtc"])),

        ("wifi" in p or "wireless data" in p or "high throughput" in p,
         any(k in s for k in ["wifi", "wlan", "unlicensed", "laa", "high throughput"])),

        ("network slicing" in p or "vehicle" in p,
         any(k in s for k in ["slicing", "v2x", "vehicle", "urllc", "23.508"])),
    ]
    return 1 if any(p_cond and s_cond for p_cond, s_cond in rules) else 0


rows = []
for p in patents:
    for s in standards:
        rows.append([p, s, assign_label(p, s)])

df = pd.DataFrame(rows, columns=["patent", "standard", "label"])
print(f"Dataset shape : {df.shape}")
print(f"Positive pairs: {df['label'].sum()}")
print(f"Negative pairs: {len(df) - df['label'].sum()}")
#Build Full Pairwise Dataset with manual labels

Dataset shape : (84, 3)
Positive pairs: 34
Negative pairs: 50


In [18]:
def evaluate_and_save(model_name: str, scores: list, df: pd.DataFrame,
                      threshold: float, output_dir: str) -> dict:
    df = df.copy()
    df["score"]     = scores
    df["predicted"] = (df["score"] >= threshold).astype(int)

    y_true = df["label"]
    y_pred = df["predicted"]

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=["Not Relevant", "Relevant"],
                yticklabels=["Not Relevant", "Relevant"])
    plt.title(model_name, fontsize=9)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    safe = model_name.replace("/", "_").replace("-", "_")
    plt.savefig(f"{output_dir}/confusion_{safe}.jpg", dpi=300)
    plt.close()

    print(f"  {model_name}  →  Acc={acc:.3f}  P={prec:.3f}  R={rec:.3f}  F1={f1:.3f}")
    return {"Model": model_name, "Accuracy": acc,
            "Precision": prec, "Recall": rec, "F1": f1}
            #Evaluation helper

In [19]:
def tfidf_similarity(df: pd.DataFrame) -> list:
    vectorizer = TfidfVectorizer()
    all_texts  = df["patent"].tolist() + df["standard"].tolist()
    vectorizer.fit(all_texts)                        # fit on all text

    scores = []
    for _, row in df.iterrows():
        pv = vectorizer.transform([row["patent"]])
        sv = vectorizer.transform([row["standard"]])
        scores.append(cosine_similarity(pv, sv)[0][0])
    return scores


def bow_similarity(df: pd.DataFrame) -> list:
    vectorizer = CountVectorizer()
    all_texts  = df["patent"].tolist() + df["standard"].tolist()
    vectorizer.fit(all_texts)

    scores = []
    for _, row in df.iterrows():
        pv = vectorizer.transform([row["patent"]])
        sv = vectorizer.transform([row["standard"]])
        scores.append(cosine_similarity(pv, sv)[0][0])
    return scores


def bm25_similarity(df: pd.DataFrame) -> list:
    """
    KEY FIX: for each patent, retrieve BM25 score against the CORRECT standard
    by matching on position within the unique standards list — not the row index.
    The score is the BM25 relevance of the patent query vs. that one standard.
    Scores are min-max normalised to [0, 1].
    """
    unique_stds = df["standard"].unique().tolist()
    std_index   = {s: i for i, s in enumerate(unique_stds)}
    tokenized   = [s.split() for s in unique_stds]
    bm25        = BM25Okapi(tokenized)

    raw = []
    for _, row in df.iterrows():
        query  = row["patent"].split()
        all_sc = bm25.get_scores(query)
        idx    = std_index[row["standard"]]
        raw.append(all_sc[idx])

    raw = np.array(raw, dtype=float)
    mn, mx = raw.min(), raw.max()
    return ((raw - mn) / (mx - mn + 1e-10)).tolist()

    #classical model helper

In [20]:
results      = []
score_storage = {}

classical_models = {
    "TF-IDF":       tfidf_similarity,
    "Bag-of-Words": bow_similarity,
    "BM25":         bm25_similarity,
}

print("=== Classical Models ===")
for name, func in classical_models.items():
    scores = func(df)
    score_storage[name] = scores
    row = evaluate_and_save(name, scores, df, THRESHOLD, output_dir)
    results.append(row)

    #running said classical models

=== Classical Models ===
  TF-IDF  →  Acc=0.595  P=0.000  R=0.000  F1=0.000
  Bag-of-Words  →  Acc=0.595  P=0.000  R=0.000  F1=0.000
  BM25  →  Acc=0.667  P=0.650  R=0.382  F1=0.481


In [21]:
print("\n=== Transformer Models ===")

unique_patents   = df["patent"].unique().tolist()
unique_standards = df["standard"].unique().tolist()

pat_idx = {p: i for i, p in enumerate(unique_patents)}
std_idx = {s: i for i, s in enumerate(unique_standards)}

for model_name in models:
    print(f"\nLoading: {model_name}")
    try:
        model = SentenceTransformer(model_name, trust_remote_code=True)

        pat_emb = model.encode(unique_patents,   convert_to_tensor=True, show_progress_bar=False)
        std_emb = model.encode(unique_standards, convert_to_tensor=True, show_progress_bar=False)

        scores = []
        for _, row in df.iterrows():
            pi = pat_idx[row["patent"]]
            si = std_idx[row["standard"]]
            scores.append(util.cos_sim(pat_emb[pi], std_emb[si]).item())

        score_storage[model_name] = scores
        result_row = evaluate_and_save(model_name, scores, df, THRESHOLD, output_dir)
        results.append(result_row)

    except Exception as e:
        print(f"  SKIPPED {model_name}: {e}")

        #running transformer models


=== Transformer Models ===

Loading: sentence-transformers/all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.txt: 0.00B [00:00, ?B/s]

  allenai/scibert_scivocab_uncased  →  Acc=0.405  P=0.405  R=1.000  F1=0.576


In [22]:
results_df = (pd.DataFrame(results)
                .sort_values("F1", ascending=False)
                .reset_index(drop=True))

results_df.to_csv(f"{output_dir}/Results_table.csv", index=False)
print("\\nFull model comparison (sorted by F1):")
print(results_df.to_string())
#building anf savinf result dataframe

\nFull model comparison (sorted by F1):
                                                          Model  Accuracy  Precision    Recall        F1
0                            sentence-transformers/gtr-t5-large  0.404762   0.404762  1.000000  0.576271
1                             sentence-transformers/gtr-t5-base  0.404762   0.404762  1.000000  0.576271
2                                        BAAI/bge-small-en-v1.5  0.404762   0.404762  1.000000  0.576271
3                                         BAAI/bge-base-en-v1.5  0.404762   0.404762  1.000000  0.576271
4                                        BAAI/bge-large-en-v1.5  0.404762   0.404762  1.000000  0.576271
5                                           intfloat/e5-base-v2  0.404762   0.404762  1.000000  0.576271
6                                          intfloat/e5-large-v2  0.404762   0.404762  1.000000  0.576271
7                              allenai/scibert_scivocab_uncased  0.404762   0.404762  1.000000  0.576271
8              

In [23]:
metrics = ["Accuracy", "Precision", "Recall", "F1"]

# --- F1 bar chart ---
plt.figure(figsize=(12, 6))
plt.barh(results_df["Model"], results_df["F1"])
plt.xlabel("F1 Score")
plt.title("Model Comparison — F1 Score")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f"{output_dir}/F1_comparison.jpg", dpi=300)
plt.close()

# --- All metrics grouped bar ---
ax = results_df.set_index("Model")[metrics].plot(kind="bar", figsize=(14, 6))
plt.title("Performance Metrics Comparison")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"{output_dir}/All_metrics_comparison.jpg", dpi=300)
plt.close()

# --- Radar chart (top 5) ---
top5   = results_df.head(TOP_N)
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False)
angles = np.concatenate([angles, [angles[0]]])

plt.figure(figsize=(6, 6))
ax = plt.subplot(111, polar=True)
for _, row in top5.iterrows():
    vals = np.concatenate([row[metrics].values, [row[metrics].values[0]]])
    ax.plot(angles, vals, label=row["Model"])
ax.set_thetagrids(angles[:-1] * 180 / np.pi, metrics)
plt.title("Top 5 Model Radar Chart")
plt.legend(loc="upper right", bbox_to_anchor=(1.45, 1.1), fontsize=7)
plt.tight_layout()
plt.savefig(f"{output_dir}/Radar_top5.jpg", dpi=300)
plt.close()

# --- Metric correlation heatmap ---
plt.figure(figsize=(6, 5))
sns.heatmap(results_df[metrics].corr(), annot=True, cmap="coolwarm")
plt.title("Metric Correlation")
plt.tight_layout()
plt.savefig(f"{output_dir}/Metric_correlation.jpg", dpi=300)
plt.close()

# --- Results table image ---
plt.figure(figsize=(12, max(4, len(results_df) * 0.4)))
plt.axis("off")
tbl = plt.table(cellText=results_df.round(3).values,
                colLabels=results_df.columns, loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.5)
plt.tight_layout()
plt.savefig(f"{output_dir}/Results_table.jpg", dpi=300)
plt.close()

print("All paper figures saved.")
#paper figures

All paper figures saved.


In [24]:
top5_names  = results_df.head(TOP_N)["Model"].tolist()
best_model  = top5_names[0]

# ROC Curve
plt.figure(figsize=(8, 6))
for name in top5_names:
    sc        = score_storage[name]
    fpr, tpr, _ = roc_curve(df["label"], sc)
    roc_auc   = auc(fpr, tpr)
    lw        = 3 if name == best_model else 1.5
    ls        = "-" if name == best_model else "--"
    plt.plot(fpr, tpr, linewidth=lw, linestyle=ls,
             label=f"{name.split('/')[-1]} (AUC={roc_auc:.3f})")
plt.plot([0, 1], [0, 1], "k:", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Top 5 Models")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{output_dir}/ROC_top5.jpg", dpi=300)
plt.close()

# Precision-Recall Curve
plt.figure(figsize=(8, 6))
for name in top5_names:
    sc                  = score_storage[name]
    precision, recall, _ = precision_recall_curve(df["label"], sc)
    lw = 3 if name == best_model else 1.5
    ls = "-" if name == best_model else "--"
    plt.plot(recall, precision, linewidth=lw, linestyle=ls,
             label=name.split("/")[-1])
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — Top 5 Models")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{output_dir}/PR_top5.jpg", dpi=300)
plt.close()

print("ROC and PR curves saved.")
#ROC & Precision-Recall Curves (top N models)

ROC and PR curves saved.


In [25]:
import pandas as pd
import numpy as np

def ranking_metrics(df: pd.DataFrame, scores: list, k_values=(1, 3, 5)):
    df = df.copy()
    df["score"] = scores

    mrr = 0.0
    recall_k  = {k: 0.0 for k in k_values}
    precision_k = {k: 0.0 for k in k_values}

    for patent, group in df.groupby("patent"):
        group = group.sort_values("score", ascending=False).reset_index(drop=True)
        relevant = set(group[group["label"] == 1].index.tolist())

        # MRR
        for rank, idx in enumerate(group.index):
            if group.loc[idx, "label"] == 1:
                mrr += 1.0 / (rank + 1)
                break

        for k in k_values:
            top_k   = set(range(min(k, len(group))))
            hits    = len(top_k & relevant)
            r_total = len(relevant)
            precision_k[k] += hits / k
            recall_k[k]    += (hits / r_total) if r_total > 0 else 0.0

    n = df["patent"].nunique()
    mrr /= n
    for k in k_values:
        precision_k[k] /= n
        recall_k[k]    /= n

    return {"MRR": mrr,
            **{f"P@{k}": precision_k[k] for k in k_values},
            **{f"R@{k}": recall_k[k]    for k in k_values}}


ranking_results = []
if 'score_storage' not in globals():
    print("Warning: 'score_storage' not found. Please ensure model evaluation cells (e.g., Classical Models, Transformer Models) have been executed.")
    score_storage = {} # Initialize as empty to prevent NameError

for name, sc in score_storage.items():
    rm = ranking_metrics(df, sc)
    ranking_results.append({"Model": name, **rm})

ranking_df = (pd.DataFrame(ranking_results)
                .sort_values("MRR", ascending=False)
                .reset_index(drop=True))

ranking_df.to_csv(f"{output_dir}/Ranking_metrics.csv", index=False)
print("\\nRanking Metrics (sorted by MRR):")
print(ranking_df.to_string())
print("\\nAll outputs saved to:", output_dir)
#Ranking Metrics (MRR, Recall@K, Precision@K)

\nRanking Metrics (sorted by MRR):
                                                          Model       MRR   P@1       P@3   P@5       R@1       R@3       R@5
0                                                        TF-IDF  0.750000  0.75  0.750000  0.70  0.080556  0.241667  0.361111
1                                                  Bag-of-Words  0.750000  0.75  0.750000  0.70  0.080556  0.241667  0.361111
2                                                          BM25  0.750000  0.75  0.666667  0.60  0.080556  0.200000  0.277778
3                        sentence-transformers/all-MiniLM-L6-v2  0.750000  0.75  0.750000  0.60  0.080556  0.241667  0.322222
4                       sentence-transformers/all-MiniLM-L12-v2  0.750000  0.75  0.583333  0.55  0.080556  0.202778  0.308333
5                       sentence-transformers/all-mpnet-base-v2  0.750000  0.75  0.666667  0.55  0.080556  0.227778  0.308333
6                    sentence-transformers/all-roberta-large-v1  0.750000  0.75  0.

In [26]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import numpy as np
!pip install rank_bm25

# =============================================================================
# Ensure results_df and ranking_df are defined for plotting
# This replicates the logic from HDGrEJAVpvBt and iQPuACk9qLPw
# =============================================================================

# Ensure 'results' is defined. If not, initialize as empty and inform the user.
if 'results' not in globals():
    print("Warning: 'results' not found. Please ensure model evaluation cells (e.g., Classical Models, Transformer Models) have been executed.")
    results = [] # Initialize as empty to prevent NameError

results_df = (pd.DataFrame(results)
                .sort_values("F1", ascending=False)
                .reset_index(drop=True))

def ranking_metrics(df: pd.DataFrame, scores: list, k_values=(1, 3, 5)):
    df = df.copy()
    df["score"] = scores

    mrr = 0.0
    recall_k  = {k: 0.0 for k in k_values}
    precision_k = {k: 0.0 for k in k_values}

    for patent, group in df.groupby("patent"):
        group = group.sort_values("score", ascending=False).reset_index(drop=True)
        relevant = set(group[group["label"] == 1].index.tolist())

        for rank, idx in enumerate(group.index):
            if group.loc[idx, "label"] == 1:
                mrr += 1.0 / (rank + 1)
                break

        for k in k_values:
            top_k   = set(range(min(k, len(group))))
            hits    = len(top_k & relevant)
            r_total = len(relevant)
            precision_k[k] += hits / k
            recall_k[k]    += (hits / r_total) if r_total > 0 else 0.0

    n = df["patent"].nunique()
    mrr /= n
    for k in k_values:
        precision_k[k] /= n
        recall_k[k]    /= n

    return {"MRR": mrr,
            **{f"P@{k}": precision_k[k] for k in k_values},
            **{f"R@{k}": recall_k[k]    for k in k_values}}

ranking_results = []

# Ensure 'score_storage' is defined. If not, initialize as empty and inform the user.
if 'score_storage' not in globals():
    print("Warning: 'score_storage' not found. Please ensure model evaluation cells (e.g., Classical Models, Transformer Models) have been executed.")
    score_storage = {} # Initialize as empty to prevent NameError

for name, sc in score_storage.items():
    rm = ranking_metrics(df, sc)
    ranking_results.append({"Model": name, **rm})

ranking_df = (pd.DataFrame(ranking_results)
                .sort_values("MRR", ascending=False)
                .reset_index(drop=True))

# =============================================================================
# CELL 14 — Comparison Charts for Paper (Original content follows)
# =============================================================================


# ── Pull live data from your pipeline results ──
# Make sure results_df and ranking_df are already defined (Cells 10 & 13)

metrics     = ["Accuracy", "Precision", "Recall", "F1"]
models_list = results_df["Model"].tolist()

# Shorten model names for readability on axes
def shorten(name):
    return name.split("/")[-1][:18]

short_names = [shorten(m) for m in models_list]


# ── CHART A: Grouped bar — Accuracy / Precision / Recall / F1 ──
x     = np.arange(len(models_list))
width = 0.2
colors = ["#4C78A8", "#F58518", "#E45756", "#72B7B2"]

fig, ax = plt.subplots(figsize=(max(14, len(models_list)*1.1), 6))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*width, results_df[metric], width, label=metric, color=color)

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(short_names, rotation=30, ha="right", fontsize=8)
ax.set_ylim(0, 1.1)
ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.set_title("Classification Metrics by Model (Acc / Prec / Rec / F1)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/chartA_classification_metrics.jpg", dpi=300)
plt.close()


# ── CHART B: Horizontal F1 bar (sorted, already sorted in results_df) ──
fig, ax = plt.subplots(figsize=(8, max(6, len(models_list)*0.45)))
colors_grad = cm.viridis(np.linspace(0.8, 0.3, len(models_list)))
ax.barh(short_names, results_df["F1"], color=colors_grad)
ax.invert_yaxis()
ax.set_xlabel("F1 Score")
ax.set_title("F1 Score Ranking — All Models (Best to Worst)")
ax.set_xlim(0, 1.05)
for i, v in enumerate(results_df["F1"]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=7)
plt.tight_layout()
plt.savefig(f"{output_dir}/chartB_f1_ranking.jpg", dpi=300)
plt.close()


# ── CHART C: Grouped bar — MRR / P@1 / P@3 / P@5 / R@5 ──
# (uses ranking_df — ensure it has these columns from Cell 13)
rank_metrics = [c for c in ranking_df.columns if c != "Model"]
x2    = np.arange(len(ranking_df))
width2 = 0.8 / len(rank_metrics)
short_rank = [shorten(m) for m in ranking_df["Model"].tolist()]

fig, ax = plt.subplots(figsize=(max(14, len(ranking_df)*1.1), 6))
for i, (metric, color) in enumerate(zip(rank_metrics, cm.tab10(np.linspace(0,1,len(rank_metrics))))):
    ax.bar(x2 + i*width2, ranking_df[metric], width2, label=metric, color=color)

ax.set_xticks(x2 + width2*(len(rank_metrics)-1)/2)
ax.set_xticklabels(short_rank, rotation=30, ha="right", fontsize=8)
ax.set_ylim(0, 1.1)
ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.set_title("Retrieval Ranking Metrics by Model (MRR / P@K / R@K)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{output_dir}/chartC_ranking_metrics.jpg", dpi=300)
plt.close()


# ── CHART D: Scatter — F1 vs MRR ──
# Merges classification and ranking results
merged = results_df[["Model","F1"]].merge(
    ranking_df[["Model","MRR"]], on="Model", how="inner"
)
merged["short"] = merged["Model"].apply(shorten)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(merged["F1"], merged["MRR"], s=70, zorder=3)
for _, row in merged.iterrows():
    ax.annotate(row["short"], (row["F1"], row["MRR"]),
                textcoords="offset points", xytext=(4, 4), fontsize=7)
ax.set_xlabel("F1 Score (Classification)")
ax.set_ylabel("MRR (Retrieval)")
ax.set_title("F1 vs MRR — Do Classification & Retrieval Metrics Agree?")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.axline((0, 0), slope=1, linestyle="--", color="gray", linewidth=0.8)
plt.tight_layout()
plt.savefig(f"{output_dir}/chartD_f1_vs_mrr.jpg", dpi=300)
plt.close()


# ── CHART E: Radar (spider) — Top 5 models, all 5 metrics ──
top5      = results_df.head(5)
top5_rank = ranking_df[ranking_df["Model"].isin(top5["Model"])].set_index("Model")
categories = ["Accuracy", "Precision", "Recall", "F1", "MRR"]
N_cat      = len(categories)
angles     = np.linspace(0, 2*np.pi, N_cat, endpoint=False).tolist()
angles    += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for _, row in top5.iterrows():
    mrr_val = top5_rank.loc[row["Model"], "MRR"] if row["Model"] in top5_rank.index else 0
    vals    = [row["Accuracy"], row["Precision"], row["Recall"], row["F1"], mrr_val]
    vals   += vals[:1]
    ax.plot(angles, vals, linewidth=1.5, label=shorten(row["Model"]))
    ax.fill(angles, vals, alpha=0.07)

ax.set_thetagrids(np.degrees(angles[:-1]), categories)
ax.set_ylim(0, 1)
ax.set_title("Top 5 Models — All Metrics (Radar)", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=7)
plt.tight_layout()
plt.savefig(f"{output_dir}/chartE_radar_top5.jpg", dpi=300)
plt.close()


# ── CHART F: Metric correlation heatmap (includes ranking metrics) ──
# Build a combined dataframe of all metrics per model
all_metrics = results_df[["Model"] + metrics].merge(
    ranking_df, on="Model", how="inner"
)
corr_cols = [c for c in all_metrics.columns if c != "Model"]
corr_mat  = all_metrics[corr_cols].corr()

fig, ax = plt.subplots(figsize=(len(corr_cols)+1, len(corr_cols)))
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm",
            ax=ax, square=True, linewidths=0.5)
ax.set_title("Metric Correlation Heatmap (All Classification + Ranking Metrics)")
plt.tight_layout()
plt.savefig(f"{output_dir}/chartF_metric_correlation.jpg", dpi=300)
plt.close()


# ── CHART G: Precision–Recall tradeoff line (top 5, sweep threshold) ──
# Uses stored scores from score_storage (Cell 9)
# Recomputes precision & recall across many thresholds

top5_names = results_df.head(5)["Model"].tolist()
thresholds = np.linspace(0.05, 0.95, 40)

fig, ax = plt.subplots(figsize=(8, 6))
for name in top5_names:
    if name not in score_storage:
        continue
    scores_arr = np.array(score_storage[name])
    y_true     = df["label"].values
    prec_vals, rec_vals = [], []
    for t in thresholds:
        y_pred = (scores_arr >= t).astype(int)
        prec_vals.append(precision_score(y_true, y_pred, zero_division=0))
        rec_vals.append(recall_score(y_true, y_pred, zero_division=0))
    ax.plot(rec_vals, prec_vals, linewidth=1.8, label=shorten(name))

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision–Recall Tradeoff — Top 5 Models (Threshold 0.05→0.95)")
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{output_dir}/chartG_pr_tradeoff.jpg", dpi=300)
plt.close()

print("All 7 comparison charts saved to:", output_dir)

All 7 comparison charts saved to: Pictures for paper
